# 04 · 聚类分析 (T7)

**输入**: `../data/train.csv`

**输出**:
- `../outputs/cluster_id_train.npy`, `../outputs/cluster_id_valid.npy`  (KMeans, 供 03 notebook 消融实验 E3 使用)
- `../outputs/gmm_cluster_id_train.npy`, `../outputs/gmm_cluster_id_valid.npy`  (GMM, 供 E4 使用)
- `../outputs/figures/04_silhouette_curve.png`
- `../outputs/figures/04_pca_clusters.png`
- `../outputs/figures/04_positive_rate_per_cluster.png`
- `../outputs/tables/04_cluster_positive_rate.csv`
- `../outputs/tables/04_kmeans_vs_gmm.csv`

**任务**: T7 聚类分析
1. 标准化 + PCA 降维到 30 维
2. KMeans (k=2..10), 用轮廓系数选最优 k
3. GMM (同样的 k 范围), 与 KMeans 对比
4. 各簇的正例比例统计
5. PCA 二维散点 + 正例率柱状图
6. 保存 cluster_id 供消融实验

In [ ]:
import sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

sys.path.insert(0, str(Path('..').resolve() / 'src'))
import data_utils
import features

warnings.filterwarnings('ignore')

OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR = OUT_DIR / 'tables';  TBL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

## 1. Load + standardize + PCA

In [ ]:
train, _ = data_utils.load_data()
X_train, X_valid, y_train, y_valid = data_utils.split_data(train)
X_train_s, X_valid_s, _, scaler = features.standardize(X_train, X_valid)
print(X_train_s.shape, X_valid_s.shape)

In [ ]:
# PCA 30 dims keeps roughly the bulk of the variance while making KMeans/GMM
# tractable. We also keep a 2-D version for the scatter plot.
pca30 = PCA(n_components=30, random_state=RANDOM_STATE).fit(X_train_s)
X_train_pca = pca30.transform(X_train_s)
X_valid_pca = pca30.transform(X_valid_s)
print('cumulative explained variance @30:', pca30.explained_variance_ratio_.sum().round(4))

pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_train_s)
X_train_2d = pca2.transform(X_train_s)

## 2. KMeans — silhouette sweep over k = 2..10

Silhouette is computed on a 20k subsample because the full 160k×30 distance matrix is too expensive. Silhouette ranking is stable under subsampling.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sub_size = min(20000, X_train_pca.shape[0])
sub_idx = rng.choice(X_train_pca.shape[0], size=sub_size, replace=False)
X_sub = X_train_pca[sub_idx]

ks = list(range(2, 11))
kmeans_records = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_full = km.fit_predict(X_train_pca)
    sil = silhouette_score(X_sub, km.predict(X_sub))
    kmeans_records.append({'k': k, 'silhouette': sil, 'inertia': km.inertia_})
    print(f'KMeans k={k}: silhouette={sil:.4f}  inertia={km.inertia_:.0f}')

kmeans_df = pd.DataFrame(kmeans_records)
best_k = int(kmeans_df.loc[kmeans_df['silhouette'].idxmax(), 'k'])
print('Best KMeans k by silhouette:', best_k)

In [ ]:
# Train final KMeans with the chosen k and produce cluster ids for train + valid.
km_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit(X_train_pca)
kmeans_train = km_final.labels_.astype('int32')
kmeans_valid = km_final.predict(X_valid_pca).astype('int32')

np.save(OUT_DIR / 'cluster_id_train.npy', kmeans_train)
np.save(OUT_DIR / 'cluster_id_valid.npy', kmeans_valid)
print('saved KMeans cluster_id arrays:', kmeans_train.shape, kmeans_valid.shape)

## 3. GMM — same k sweep

In [ ]:
gmm_records = []
for k in ks:
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                          random_state=RANDOM_STATE, n_init=1, max_iter=200)
    gmm.fit(X_train_pca)
    labels_sub = gmm.predict(X_sub)
    # silhouette only meaningful when ≥2 distinct labels in the subsample
    sil = silhouette_score(X_sub, labels_sub) if len(np.unique(labels_sub)) > 1 else float('nan')
    gmm_records.append({'k': k, 'silhouette': sil, 'bic': gmm.bic(X_train_pca)})
    print(f'GMM k={k}: silhouette={sil:.4f}  BIC={gmm.bic(X_train_pca):.0f}')

gmm_df = pd.DataFrame(gmm_records)
best_k_gmm = int(gmm_df.loc[gmm_df['silhouette'].idxmax(), 'k'])
print('Best GMM k by silhouette:', best_k_gmm)

In [ ]:
gmm_final = GaussianMixture(n_components=best_k_gmm, covariance_type='full',
                            random_state=RANDOM_STATE, max_iter=300).fit(X_train_pca)
gmm_train = gmm_final.predict(X_train_pca).astype('int32')
gmm_valid = gmm_final.predict(X_valid_pca).astype('int32')
np.save(OUT_DIR / 'gmm_cluster_id_train.npy', gmm_train)
np.save(OUT_DIR / 'gmm_cluster_id_valid.npy', gmm_valid)
print('saved GMM cluster_id arrays:', gmm_train.shape, gmm_valid.shape)

## 4. KMeans vs GMM summary table

In [ ]:
compare = pd.DataFrame({
    'method':       ['KMeans', 'GMM'],
    'best_k':       [best_k, best_k_gmm],
    'silhouette':   [kmeans_df['silhouette'].max(), gmm_df['silhouette'].max()],
    'criterion':    ['inertia', 'BIC'],
    'criterion_value': [
        float(kmeans_df.loc[kmeans_df['k'] == best_k, 'inertia'].iloc[0]),
        float(gmm_df.loc[gmm_df['k'] == best_k_gmm, 'bic'].iloc[0]),
    ],
})
compare.to_csv(TBL_DIR / '04_kmeans_vs_gmm.csv', index=False)
compare

## 5. Silhouette curves (KMeans + GMM)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(kmeans_df['k'], kmeans_df['silhouette'], '-o', label='KMeans')
ax.plot(gmm_df['k'], gmm_df['silhouette'], '-s', label='GMM', color='C2')
ax.axvline(best_k,     color='C0', linestyle='--', alpha=0.5)
ax.axvline(best_k_gmm, color='C2', linestyle='--', alpha=0.5)
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score (20k subsample)')
ax.set_title('Silhouette score vs k')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / '04_silhouette_curve.png', dpi=150)
plt.show()

## 6. Per-cluster positive rate (KMeans)

In [ ]:
summary_rows = []
for cluster_id in np.unique(kmeans_train):
    mask = kmeans_train == cluster_id
    n = int(mask.sum())
    pos = int(y_train.values[mask].sum())
    summary_rows.append({
        'cluster': int(cluster_id),
        'n_samples': n,
        'n_positive': pos,
        'positive_rate': pos / n if n else 0.0,
    })
cluster_summary = pd.DataFrame(summary_rows).sort_values('positive_rate', ascending=False)
cluster_summary.to_csv(TBL_DIR / '04_cluster_positive_rate.csv', index=False)
cluster_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
order = cluster_summary['cluster'].astype(str)
ax.bar(order, cluster_summary['positive_rate'],
       color=sns.color_palette('flare', n_colors=len(cluster_summary)))
ax.axhline(float(y_train.mean()), color='gray', linestyle='--', label=f'overall pos rate = {float(y_train.mean()):.3f}')
for i, (_, row) in enumerate(cluster_summary.iterrows()):
    ax.text(i, row['positive_rate'] + 0.002, f"{row['positive_rate']:.3f}", ha='center', fontsize=9)
ax.set_xlabel('KMeans cluster id')
ax.set_ylabel('positive rate (target=1)')
ax.set_title('Positive rate per KMeans cluster')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / '04_positive_rate_per_cluster.png', dpi=150)
plt.show()

## 7. PCA 2-D scatter (subsample for legibility)

In [ ]:
scatter_size = min(8000, X_train_2d.shape[0])
scatter_idx = rng.choice(X_train_2d.shape[0], size=scatter_size, replace=False)
scatter_pts = X_train_2d[scatter_idx]
scatter_lab = kmeans_train[scatter_idx]

fig, ax = plt.subplots(figsize=(7, 5.5))
palette = sns.color_palette('tab10', n_colors=int(np.unique(scatter_lab).size))
for cid, color in zip(np.unique(scatter_lab), palette):
    m = scatter_lab == cid
    ax.scatter(scatter_pts[m, 0], scatter_pts[m, 1], s=6, alpha=0.5, label=f'cluster {cid}', color=color)
ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
ax.set_title(f'KMeans (k={best_k}) — PCA 2D projection (subsample {scatter_size})')
ax.legend(markerscale=2)
fig.tight_layout()
fig.savefig(FIG_DIR / '04_pca_clusters.png', dpi=150)
plt.show()

## 8. 报告写作要点 (供第 3/4 章引用)

- KMeans 在 PCA-30 空间下的轮廓系数最优 k 与 GMM 对比写进报告; 通常两者 silhouette 都偏低，说明 Santander 没有强自然分群结构。
- 各簇正例率与全局正例率 (~10%) 接近，说明聚类几乎没有捕获 target 信号——后续在 03 notebook 的 E3/E4 实验中也会得到几乎不变的 ROC-AUC。
- cluster_id 已落盘，03 notebook 的 E3/E4 自动消费，无需再跑。